In [ ]:
# how to uncover truths that don't matter - second section

In [ ]:
# loading libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
import time

In [ ]:
# import functions
import coriolis_functions 
# import selfbuild module
import coriolis_module
# check out if import worked as expected
print(dir(coriolis_module))
print(dir(coriolis_functions))

In [ ]:
fdf = pd.read_csv("data/flights_sample_3m.csv")
ldf = pd.read_csv("data/airports.csv")

print(len(fdf), " : number of flights in dataframe")
print(len(ldf), " : number of airport-locations in dataframe")

print(fdf.columns)

In [ ]:
fdf = fdf.drop( ['AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE',
       'FL_NUMBER', 'ORIGIN_CITY', 'DEST_CITY',
       'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT',
       'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY',
       'CANCELLATION_CODE', 'CRS_ELAPSED_TIME',
       'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER',
       'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
       'DELAY_DUE_LATE_AIRCRAFT'] ,axis=1)

In [ ]:
fdf.dtypes

In [ ]:
# convert to datetime
fdf["FL_DATE"] = pd.to_datetime(fdf["FL_DATE"])

In [ ]:
# setting up boolean columns
boolean_columns = ["CANCELLED", "DIVERTED"]
fdf[boolean_columns] = fdf[boolean_columns].astype("bool")

In [ ]:
fdf = fdf[~fdf["CANCELLED"]]
fdf = fdf.drop(["CANCELLED"], axis=1)

In [ ]:
# defining a helper function
def convert_time(df, time_columns):
    """ Convert integer time columns to HH:MM format """
    for col in time_columns:
        # Handle missing values and convert times
        df[col] = df[col].fillna(0).astype(int).apply(lambda x: f"{x//100:02d}:{x%100:02d}")
        # Adjust for hours == 24
        df[col] = df[col].replace("24:00", "00:00")
    return df

# applying the time conversion
fdf = convert_time(fdf, ["WHEELS_OFF", "WHEELS_ON"])

fdf["woff_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["WHEELS_OFF"])
fdf["won_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["WHEELS_ON"])

# dropping original time columns
fdf = fdf.drop(["WHEELS_OFF", "WHEELS_ON"], axis=1)

In [ ]:
# checking unique values in both datasets
fdf_airports = set(fdf["ORIGIN"].unique()).union(set(fdf["DEST"].unique()))
ldf_airports = set(ldf["IATA"].unique())  

# finding missing airport codes in ldf
missing_airports = fdf_airports.difference(ldf_airports)

# dropping rows where "ORIGIN" or "DEST" are in missing_airports
fdf = fdf[~fdf["ORIGIN"].isin(missing_airports) & ~fdf["DEST"].isin(missing_airports)]

# merging fdf with ldf to add geographical data for ORIGIN and DEST
fdf = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="ORIGIN", right_on="IATA", how="left")

fdf = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="DEST", right_on="IATA", how="left", suffixes=("_ORIGIN", "_DEST"))

# dropping original origin and destination columns
fdf = fdf.drop(["ORIGIN", "DEST"], axis=1)

# enforcing categories on newly generated columns
fdf[["IATA_ORIGIN", "IATA_DEST"]] = fdf[["IATA_ORIGIN", "IATA_DEST"]].astype("category")

In [ ]:
fdf["airtime"] = fdf["won_datetime"] - fdf["woff_datetime"]

In [ ]:
fdf = fdf[(fdf["airtime"].dt.total_seconds() > 0)]

In [ ]:
# calculating haversinedistance
fdf_merged['haversine_distance'] = coriolis_functions.haversine(
    fdf_merged["LATITUDE_ORIGIN"],
    fdf_merged["LONGITUDE_ORIGIN"],
    fdf_merged["LATITUDE_DEST"], 
    fdf_merged["LONGITUDE_DEST"]
)

In [ ]:
start_time = time.time()

In [ ]:
# Apply direction vector function row-wise
directions = fdf_merged.apply(
    lambda row: coriolis_functions.direction_vector(
        row["LATITUDE_ORIGIN"], 
        row["LONGITUDE_ORIGIN"], 
        row["LATITUDE_DEST"], 
        row["LONGITUDE_DEST"]
    ), axis=1
)

# Split the resulting direction vectors into separate columns
fdf_merged["x_direction"] = directions.apply(lambda x: x[0])
fdf_merged["y_direction"] = directions.apply(lambda x: x[1])
fdf_merged["z_direction"] = directions.apply(lambda x: x[2])

In [ ]:
# Apply the calculate_total_drift function row by row
fdf_merged["total_drift_distance"] = fdf_merged.apply(
    lambda row: coriolis_functions.calculate_total_drift(
        row, row["airtime"].total_seconds(), num_steps=100  # adjust num_steps for precision
    ), axis=1
)

In [ ]:
end_time = time.time()
print(f"{end_time - start_time:,.2f} : calculation time in seconds")

In [ ]:
# checking out the absolut drift distance
fdf["total_drift_distance"].describe()

In [ ]:
# division of drift-distance through distance eliminates dimension 
# so we have a unit-less dirft-factor
fdf.loc[:, "drift_factor"] = fdf["total_drift_distance"] / fdf["haversine_distance"].replace(0, pd.NA)

In [ ]:
# taking a look at drift-factor
fdf["drift_factor"].describe()

In [ ]:
# plotting scatterplot of haversine-distance and total-drift-distance
plt.figure(figsize=(10, 6))
sns.scatterplot(x="haversine_distance", y="total_drift_distance", data=fdf, alpha=0.5)
plt.title("Scatterplot: Relation between haversine-distance and total-drift-distance")
plt.xlabel("Haversine-distance [km]")
plt.ylabel("Total-drift-distance [km]")
plt.axhline(0, color="red", linestyle="--")  # Reference line for on-time arrivals
plt.axvline(0, color="blue", linestyle="--")  # Reference line for on-time departures
plt.grid(True)
plt.show()

In [ ]:
# plotting scatterplot of relation between harversine-distance and dirft-factor
plt.figure(figsize=(10, 6))
sns.scatterplot(x="haversine_distance", y="drift_factor", data=fdf, alpha=0.5)
plt.title("Scatterplot: Relation between haversine-distance and drift-factor")
plt.xlabel("Haversine-distance [km]")
plt.ylabel("Drift-factor := (total-drift-distance [km]/Haversine-distance [km])")
plt.axhline(0, color="red", linestyle="--")  # Reference line for on-time arrivals
plt.axvline(0, color="blue", linestyle="--")  # Reference line for on-time departures
plt.grid(True)
plt.show()

In [ ]:
# setting up boxplot

# prepairing data for boxplots to take a view about outliers when it comes to drift-distances
melted_data = fdf_merged.melt(value_vars=["total_drift_distance"], var_name="Metric", value_name="Kilometer")

# plotting
plt.figure(figsize=(3,6))
sns.boxplot(x="Metric", y="Kilometer", data=melted_data, legend=False)
plt.title("Boxplot of distances")

In [ ]:
# how much is this coriolis-drift in absolute numbers? 
absolute_drift_distance = fdf["total_drift_distance"].sum()

# lets compare that somehow to the distances used for calculations
absolute_travel_distance = fdf["haversine_distance"].sum()

# since we assume haversine_distance >= total_drift_distance 
interim_results_dict = {
     "total drift distance [km]" : f"{absolute_drift_distance:.2f}",
    "total traveled distance by non cancelled flights [km]" : f"{absolute_travel_distance:.2f}",
    "average-drift-factor": f"{(absolute_drift_distance / absolute_travel_distance):.4f}",
    "%-value of average-coriolis-drift in dataframe": f"{ (absolute_drift_distance / absolute_travel_distance)*100:.2f}"
}

for name in interim_results_dict:
    print(f"{interim_results_dict[name]} : {name}")

In [ ]:
print(f"{len(fdf):,} : Rows in dataset at time of evaluation.")